In [0]:
# ============================================================
# Phase 10.3 / 15.1 — SQL Generator
# ============================================================

import json
import re
from datetime import datetime


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DEFAULT_TABLE = "genai_copilot.gold.region_sales"

ALLOWED_TABLES = [
    "genai_copilot.gold.region_sales",
    "genai_copilot.silver.sales",
]


# ------------------------------------------------------------
# SQL Generation
# ------------------------------------------------------------

def generate_sql_request(question):
    """
    Convert a natural-language analytical question
    into a structured SQL generation request.

    This function does NOT execute SQL.
    """

    if not question or not question.strip():
        return {
            "success": False,
            "question": question,
            "sql": None,
            "table": None,
            "error": "Question cannot be empty."
        }

    normalized_question = question.strip().lower()

    # --------------------------------------------------------
    # Example analytical patterns
    # --------------------------------------------------------

    # Highest revenue by region
    if (
        "highest revenue" in normalized_question
        and "region" in normalized_question
    ):
        sql = """
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
ORDER BY total_revenue DESC
LIMIT 10
""".strip()

        return {
            "success": True,
            "question": question,
            "normalized_question": normalized_question,
            "sql": sql,
            "table": "genai_copilot.gold.region_sales",
            "generation_method": "rule_based",
            "generated_at": datetime.now()
        }

    # --------------------------------------------------------
    # Total revenue
    # --------------------------------------------------------

    if (
        "total revenue" in normalized_question
        or "overall revenue" in normalized_question
    ):
        sql = """
SELECT
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
""".strip()

        return {
            "success": True,
            "question": question,
            "normalized_question": normalized_question,
            "sql": sql,
            "table": "genai_copilot.gold.region_sales",
            "generation_method": "rule_based",
            "generated_at": datetime.now()
        }

    # --------------------------------------------------------
    # Revenue by region
    # --------------------------------------------------------

    if (
        "revenue by region" in normalized_question
        or "region revenue" in normalized_question
    ):
        sql = """
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
ORDER BY total_revenue DESC
""".strip()

        return {
            "success": True,
            "question": question,
            "normalized_question": normalized_question,
            "sql": sql,
            "table": "genai_copilot.gold.region_sales",
            "generation_method": "rule_based",
            "generated_at": datetime.now()
        }

    # --------------------------------------------------------
    # Unsupported question
    # --------------------------------------------------------

    return {
        "success": False,
        "question": question,
        "normalized_question": normalized_question,
        "sql": None,
        "table": None,
        "generation_method": "none",
        "generated_at": datetime.now(),
        "error": (
            "No SQL generation pattern is available for this question."
        )
    }


# ------------------------------------------------------------
# SQL Request Validation
# ------------------------------------------------------------

def validate_sql_request(sql_request):
    """
    Validate the structure returned by generate_sql_request().
    """

    if not isinstance(sql_request, dict):
        return {
            "valid": False,
            "error": "SQL request must be a dictionary."
        }

    if not sql_request.get("success"):
        return {
            "valid": False,
            "error": sql_request.get(
                "error",
                "SQL generation failed."
            )
        }

    sql = sql_request.get("sql")

    if not sql:
        return {
            "valid": False,
            "error": "Generated SQL is empty."
        }

    return {
        "valid": True,
        "error": None
    }


# ------------------------------------------------------------
# Test Helper
# ------------------------------------------------------------

def inspect_sql_request(question):
    """
    Generate SQL and return a compact inspection object.
    """

    result = generate_sql_request(question)

    validation = validate_sql_request(result)

    return {
        "question": question,
        "generation_success": result.get("success"),
        "sql": result.get("sql"),
        "valid": validation["valid"],
        "error": (
            result.get("error")
            or validation.get("error")
        )
    }


print("SQL Generator loaded successfully.")
print(
    "generate_sql_request:",
    callable(generate_sql_request)
)
print(
    "validate_sql_request:",
    callable(validate_sql_request)
)